In [ ]:
import marimo as mo

# RAG with QueryChat: NYC Taxi Demo

**The problem querychat solves without RAG:** the LLM knows the column *names* and
value *ranges* from the data schema, but not what they *mean*. We can use data description in querychat to put the additional description in systems prompt, but what if it is huge and odd (as almost any rulebook)?
Well, ideally we would want to inject only the portion relevant to each query with local information, and for add it to the query. This is where Retrieval-Augmented Generation knocks to our doors.

In our example, LLM sees `RatecodeID` values range from 1–6, but doesn't know that 2 = JFK flat rate.

**RAG fixes this:** for every user question, retrieve the relevant domain knowledge chunks
and inject them into the user message before it reaches the LLM.
Different questions -> different chunks -> the right (hopefully, search is not perfect, and knowledge base can be incomplete) context every time.

```mermaid
flowchart LR
    Q([User question]) --> R[TF-IDF retrieve<br/>top-k chunks]
    KB[(Knowledge<br/>Base)] --> R
    R --> A[Augmented<br/>message]
    A --> LLM[LLM]
    LLM --> SQL[SQL]
    SQL --> DB[(DuckDB)]
    DB --> Ans([Answer])
```


**RAG retrieval subprocess** — what happens inside the "retrieve" step:

```mermaid
flowchart LR
    Q([User query]) --> EQ["Embed query<br/>(same method as docs)"]
    DOCS[("Doc embeddings<br/>(pre-computed)")] --> COS
    EQ --> COS["Cosine similarity<br/>query ↔ each chunk"]
    COS --> RANK[Rank by score]
    RANK --> THR{"score ≥<br/>threshold?"}
    THR -->|"yes → top-k"| CTX[Selected chunks]
    THR -->|no matches| SKIP[No injection]
    CTX --> AUG[Inject into<br/>user prompt]
```

Three key steps: **(1)** vectorize the query with the same method used to index documents,
**(2)** compute similarity against every stored chunk vector, **(3)** rank and select the top-k
chunks above a score threshold — only those get injected into the prompt.

**What we'll build:**

1. `QueryChat` without RAG: see what the LLM knows from schema alone
2. TF-IDF knowledge base: retrieve relevant chunks for any query
3. Per-query injection: augment the user message
4. Side-by-side: same questions with vs without domain knowledge
5. Tool loop walkthrough
6. How to wire this into a Shiny app

In [ ]:
import os
import chatlas as ctl
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv, find_dotenv
from querychat import QueryChat

# Load .env — try cwd, cwd/.., and cwd/rag/ to handle running from repo root or rag/
_cwd = Path.cwd()
for _p in [_cwd / ".env", _cwd / "rag" / ".env", _cwd.parent / ".env"]:
    if _p.exists():
        load_dotenv(_p)
        break

---
## A. QueryChat without RAG

Let's see what the LLM knows *before* we inject any domain knowledge.
We sample 50 000 rows — enough for meaningful queries, fast for DuckDB.

In [ ]:
taxi = pd.read_parquet("../data/taxi_2024-01.parquet").sample(50_000, random_state=42)
print(f"Loaded {len(taxi):,} rows × {len(taxi.columns)} columns")

In [ ]:
qc_base = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)

In [ ]:
print(qc_base.system_prompt)

**Notice:**
- The `<database_schema>` block lists column names, types, and value *ranges*
  (e.g. `RatecodeID`: 1.0–99.0), but no meaning for those codes.
- There's no `<data_description>` block, it only appears when you pass one.
- The LLM will have to guess that `RatecodeID=2` might be airport-related.

Let's confirm: ask a question that requires domain knowledge.

In [ ]:
_client = qc_base.client()
_response = _client.chat(
    "How do I filter for JFK airport trips? What column and value should I use?",
    echo="none",
)
print("WITHOUT RAG:", _response)

---
## B. Building the Knowledge Base

Our KB is plain `.txt` — paragraphs separated by blank lines, one topic per paragraph.
This is the simplest possible RAG KB format.

**TF-IDF retrieval:** no API key needed, no model download.
It works on term overlap — good enough for structured domain glossaries.
(For semantic similarity across natural language, you'd use sentence-transformers or an embedding API.)

In [ ]:
_kb_path = Path("../rag/knowledge_base/taxi_glossary.txt")
kb_chunks = [c.strip() for c in _kb_path.read_text().split("\n\n") if c.strip()]
kb_vectorizer = TfidfVectorizer()
kb_vectors = kb_vectorizer.fit_transform(kb_chunks)
print(f"Loaded {len(kb_chunks)} chunks → TF-IDF matrix shape: {kb_vectors.shape}")

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Return top_k most relevant chunks for query using TF-IDF cosine similarity."""
    q_vec = kb_vectorizer.transform([query])
    scores = cosine_similarity(q_vec, kb_vectors).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [kb_chunks[i] for i in top_idx if scores[i] > 0]

print("Query: 'how are airport trips identified?'")
for _c in retrieve("how are airport trips identified?"):
    print(" ·", _c[:80])

---

### TF-IDF vectors ≠ embeddings

TF-IDF *does* produce vectors — but they are **sparse word-frequency vectors**, not embeddings.
Each dimension corresponds to one word in the vocabulary. Most values are 0 (word absent),
a few are non-zero (word present, weighted by how rare it is across all chunks).

**Embedding vectors** are *dense* — every dimension is non-zero, learned by a neural network
to encode *meaning*, not just word presence. Similar sentences end up close in space even
with completely different words.

Run the next cell to see what a TF-IDF vector actually looks like for one of our taxi chunks.

In [ ]:
_vec = kb_vectorizer.transform([kb_chunks[0]]).toarray()[0]
_vocab = kb_vectorizer.get_feature_names_out()

# Show only non-zero dimensions (the rest are 0)
_nonzero = [(w, round(float(v), 3)) for w, v in zip(_vocab, _vec) if v > 0]
_nonzero.sort(key=lambda x: -x[1])

print(f"Chunk: \"{kb_chunks[0][:80]}...\"")
print(f"\nVector has {len(_vec)} dimensions total, {len(_nonzero)} non-zero:")
for _word, _score in _nonzero:
    bar = "█" * int(_score * 30)
    print(f"  {_word:20s} {_score:.3f}  {bar}")
print(f"\nCompare: an embedding vector (e.g. all-MiniLM-L6-v2) has 384 dimensions, ALL non-zero.")

---

**Optional: try real embeddings with Jina AI** — free tier, no credit card, no model download.

Get a free API key at [jina.ai](https://jina.ai/) (1M tokens free), set `JINA_API_KEY` in your `.env`,
then run the next cell to see a real 1024-dim dense embedding vector for the same chunk.

In [ ]:
import requests as _requests
import numpy as _np

_JINA_KEY = os.getenv("JINA_API_KEY", "")

if not _JINA_KEY:
    print("⚠️  Set JINA_API_KEY in your .env to run this cell.")
    print("   Get a free key (1M tokens, no credit card) at: https://jina.ai/")
else:
    _chunk = kb_chunks[0]
    _resp = _requests.post(
        "https://api.jina.ai/v1/embeddings",
        headers={"Authorization": f"Bearer {_JINA_KEY}", "Content-Type": "application/json"},
        json={"model": "jina-embeddings-v3", "input": [_chunk], "task": "retrieval.passage"},
    )
    _emb = _resp.json()["data"][0]["embedding"]
    _arr = _np.array(_emb)

    print(f"Chunk: \"{_chunk[:80]}...\"")
    print(f"\nEmbedding: {len(_arr)} dimensions, all non-zero")
    print(f"  First 8 values: {_arr[:8].round(4)}")
    print(f"  Min: {_arr.min():.4f}  Max: {_arr.max():.4f}  Non-zero: {(_arr != 0).sum()}/{len(_arr)}")
    print(f"\nThis dense vector encodes *meaning* — similar sentences will be")
    print(f"close in this 1024-dim space even with completely different words.")

---

### Our demo vs production — the upgrade path

**In our demo** we use TF-IDF on word-frequency vectors. It gives us reasonable matching
— and it's not bad at all for term-heavy, small glossaries where exact keywords appear in
both the query and the knowledge base. No API key, no model download, runs in < 1ms.

But every design choice we made has a production-grade alternative. Here's when you'd
reach for each one:

---

**When I want semantic matching** — similar *meaning* matches, not just exact words:

I use **embeddings**. Both query and documents are encoded as dense vectors so that
similar sense is close in vector space, even without exact word overlap. "JFK airport fare"
matches "flat rate to Kennedy" — TF-IDF would miss this completely.

Which embedding approach to use depends on your constraints:

---

**When I want free local embeddings with no API key and no internet** ->
I use [`sentence-transformers`](https://www.sbert.net/) — download once, run forever:

- [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) — ~90 MB, ~5 ms/query on CPU, good general purpose
- [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5) — 130 MB, top-ranked on MTEB benchmark

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(kb_chunks)  # 384-dim dense vectors
```

---

**When I want cheap API embeddings with no model download** ->
I use [Jina AI](https://jina.ai/) — free tier (1M tokens, no credit card), 1024-dim vectors,
great for prototypes and classroom demos. We just tested this above!

```python
import requests
resp = requests.post(
    "https://api.jina.ai/v1/embeddings",
    headers={"Authorization": f"Bearer {JINA_API_KEY}"},
    json={"model": "jina-embeddings-v3", "input": texts, "task": "retrieval.passage"},
)
vectors = [d["embedding"] for d in resp.json()["data"]]
```

---

**When I need production-grade embeddings** ->
I use [OpenAI `text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings)
(~$0.00002/1K tokens, 1536-dim) or [Cohere Embed v3](https://docs.cohere.com/docs/embed-2)
(strong multilingual support). Both require an API key and paid account.

📖 Tutorial: [LangChain RAG quickstart](https://python.langchain.com/docs/tutorials/rag/)

---

**When I need to store and search a large collection of documents:**

In our demo, we keep all chunk vectors in a NumPy array in memory — fine for 20 chunks,
impractical for 20,000 documents. A **vector database** stores document text + embedding
together and uses approximate nearest-neighbor (ANN) index for fast similarity search at scale.

| Vector DB | Best for | Link |
|---|---|---|
| [ChromaDB](https://docs.trychroma.com/) | Python prototyping, local or embedded | [Quickstart](https://docs.trychroma.com/docs/overview/getting-started) |
| [MongoDB Atlas Vector Search](https://www.mongodb.com/products/platform/atlas-vector-search) | Already on MongoDB | [Tutorial](https://www.mongodb.com/docs/atlas/atlas-vector-search/tutorials/) |
| [Pinecone](https://www.pinecone.io/) | Fully managed, production-grade | [Quickstart](https://docs.pinecone.io/guides/get-started/quickstart) |
| [Weaviate](https://weaviate.io/) | Open source, hybrid + multi-modal search | [Quickstart](https://weaviate.io/developers/weaviate/quickstart) |
| [Qdrant](https://qdrant.tech/) | High-performance, Rust-based | [Tutorial](https://qdrant.tech/documentation/quickstart/) |

---

**When I use expensive embedding API calls, or have a lot of documents:**

In our demo, we compute TF-IDF vectors on the fly every time. With paid embedding APIs,
that means paying per document on every restart. Instead: **pre-calculate vectors at index time**
and store them — you pay for each document once. Vector databases handle this natively:
index once → query many times → pay only for query embeddings.

---

**When I prototype with many queries and want to save money:**

I use a **cheap open-source LLM** instead of GPT-4. [Groq](https://console.groq.com/) offers
free-tier inference with fast open-source models (Llama 3, Gemma 2). Same RAG pipeline,
fraction of the cost.

> **Note:** Groq provides LLM inference only — no embedding API. For the retrieval step,
> keep using local `sentence-transformers` (free, no API key). Only the generation step
> (answering the question) moves to Groq.

📖 Tutorial: [Groq docs](https://console.groq.com/docs/overview) |
[LlamaIndex starter](https://docs.llamaindex.ai/en/stable/getting_started/starter_example/)

---

**When I need explainable/auditable retrieval** — I must explain *why* a chunk was selected:

I use **BM25** — a keyword scoring algorithm (used in Elasticsearch, Lucene). No dense
embedding vectors — just term frequency counts, fully transparent scores, easy to debug:
"this chunk matched because it has 3 occurrences of 'JFK' and 2 of 'airport'."
Great for compliance, code reviews, and stakeholder demos.

📦 [`rank_bm25`](https://pypi.org/project/rank-bm25/) — pure-Python, drop-in replacement for TF-IDF

---

**When my documents have structured metadata** (dates, categories, authors alongside text):

I use **hybrid search** — vector similarity + metadata filters together. "Find chunks about
airport fares written after 2023" = semantic similarity on content + filter on date field.

Most vector DBs support this natively: ChromaDB [`where` clause](https://docs.trychroma.com/docs/querying-collections/metadata-filtering),
Weaviate [hybrid search](https://weaviate.io/developers/weaviate/search/hybrid),
Pinecone [metadata filtering](https://docs.pinecone.io/guides/data/filter-with-metadata).

---

**When I want to evaluate how well my retrieval works:**

RAG can fail silently — wrong chunks retrieved, right chunks missed. Use an eval framework:

- [RAGAS](https://docs.ragas.io/) — measures faithfulness, answer relevancy, context precision
- [LangSmith](https://docs.smith.langchain.com/) — trace and evaluate RAG chains end-to-end

---

**When my users write queries in different languages:**

TF-IDF and English-only models will miss cross-language matches. Use **multilingual embeddings**:

- [`intfloat/multilingual-e5-base`](https://huggingface.co/intfloat/multilingual-e5-base) — free, 100+ languages
- [Cohere Embed v3](https://docs.cohere.com/docs/multilingual-language-models) — API, strong multilingual performance

---
## C. Per-query RAG injection

The trick: inject retrieved context into the **user message**.
This way every question gets its own relevant chunks — true per-query retrieval.

```python
def chat_with_rag(client, query):
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\\n\\n".join(chunks)
        query = f"Relevant domain context:\\n{context}\\n\\nQuestion: {query}"
    return client.chat(query, echo="none")
```

`QueryChat` is built once with no `data_description` — it stays generic.
Retrieval happens fresh for every `.chat()` call.

In [ ]:
qc_rag = QueryChat(
    taxi,
    "taxi",
    client=ctl.ChatGithub(model="gpt-4.1-mini", api_key=os.getenv("GITHUB_TOKEN")),
)

In [ ]:
def chat_with_rag(client, query: str) -> str:
    chunks = retrieve(query, top_k=3)
    if chunks:
        context = "\n\n".join(chunks)
        query = f"Relevant domain context:\n{context}\n\nQuestion: {query}"
    return client.chat(query, echo="none")

---
## D. Side-by-side: queries that need domain knowledge

These queries require knowing what the *codes* mean: exactly where RAG helps.

**With RAG — the retrieval subprocess on a real query:**

```mermaid
flowchart LR
    Q(["🔍 filter JFK trips?"]) --> EQ["Embed query<br/>(TF-IDF vector)"]
    KB[("KB chunk vectors<br/>(pre-built)")] --> COS
    EQ --> COS["Cosine similarity<br/>query ↔ 20 chunks"]
    COS --> RANK["Rank scores:<br/>chunk 7 → 0.82<br/>chunk 3 → 0.41<br/>chunk 12 → 0.05"]
    RANK --> TOP["Top match (0.82):<br/>RatecodeID 2 = JFK"]
    TOP --> AUG["Augmented prompt:<br/>context + question"]
    AUG --> LLM[LLM]
    LLM --> SQL["WHERE RatecodeID = 2 ✓"]
```

**Without RAG:** same question goes straight to LLM with no domain context → LLM guesses → `WHERE RatecodeID = 99 ❌`

In [ ]:
Q1 = [
    "How do I filter for JFK airport trips? What column and value should I use?",
]

for _q in Q1:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

In [ ]:
Q2 = [
    "What share of trips used a negotiated fare rate?",
]

for _q in Q2:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

In [ ]:
Q3 = [
    "Compare average tip for credit card vs cash payments — which columns encode this?",
]

for _q in Q3:
    print(f"❓ {_q}")
    print("  WITHOUT RAG:", qc_base.client().chat(_q, echo="none"))
    print("-" * 80)
    print("  WITH RAG:   ", chat_with_rag(qc_rag.client(), _q))
    print("=" * 80)

---
## G. When to use RAG — and when not to

**✅ Use RAG when:**
- Your domain has structured codes, acronyms, or jargon the LLM doesn't know
- The knowledge base changes frequently — update the KB, not the model
- You need to inject only the relevant subset of a large rulebook or glossary
- You want to keep costs low — no fine-tuning, no giant system prompt
- Your data has a schema the LLM can see but *values* that need human explanation

**❌ Skip RAG when:**
- The LLM already knows the domain well (standard SQL, Python, common APIs)
- Your entire knowledge base is tiny (< 10 paragraphs) — just put it all in the system prompt via `data_description`
- You need guaranteed accuracy — RAG retrieval can fail silently (wrong chunks, missed chunks)
- A simple few-shot prompt with examples would suffice

**⚠️ RAG failure modes to watch for:**
- **Retrieval miss**: the right chunk isn't retrieved (vocabulary mismatch, threshold too high)
- **Hallucination despite context**: LLM ignores injected chunks and makes up an answer anyway
- **Context window overflow**: too many chunks injected → LLM loses focus on the question
- **Stale KB**: knowledge base not updated → outdated or wrong context injected silently

**📊 RAG is a spectrum, not a binary choice:**

| Level | What you do | Example |
|---|---|---|
| 0 — No RAG | Everything in system prompt | `data_description="RatecodeID 2 = JFK"` |
| 1 — Simple RAG | TF-IDF + flat files (our demo) | 20 glossary paragraphs, < 1ms |
| 2 — Embedding RAG | Sentence-transformers + vector DB | 10K docs, semantic search |
| 3 — Agentic RAG | LLM decides *when* and *what* to retrieve | Multi-step reasoning, tool use |

Start at level 0. Move up only when you hit the wall at the current level.

---
## E. Watching the tool loop

Same as the querychat notebook from the last lecture. The augmented query is what the LLM sees; the tool loop is unchanged.

In [ ]:
_step = [0]

def _on_req(req):
    _step[0] += 1
    print(f"── Step {_step[0]}: LLM requests tool ──")
    print(f"   Tool: {req.name}  Args: {req.arguments}")

def _on_res(res):
    _step[0] += 1
    print(f"── Step {_step[0]}: Tool returns result ──")
    print(f"   {str(res.value)[:300]}")

_client = qc_rag.client()
_client.on_tool_request(_on_req)
_client.on_tool_result(_on_res)

_query = "What is the average fare for JFK trips vs all trips?"
_chunks = retrieve(_query, top_k=3)
_augmented = f"Relevant domain context:\n{chr(10).join(_chunks)}\n\nQuestion: {_query}" if _chunks else _query

print("── Step 0: User sends message ──────────────────────")
print("   Original query:", _query)
print("   Augmented with RAG context:")
print("   " + _augmented[:400].replace("\n", "\n   "))
print()
_response = _client.chat(_augmented, echo="none")
_step[0] += 1
print(f"── Step {_step[0]}: LLM final response ──")
print(_response)

---
## F. Wiring into a Shiny App

Everything you saw maps directly to a Shiny app:

- `retrieve()` + TF-IDF index **module level** (built once, shared across sessions)
- `QueryChat(...)` → **module level** too — no `data_description`, stays generic
- `chat_with_rag(client, query)` wraps the per-message send inside the reactive effect

```python
# Module level
qc = QueryChat(df, "taxi", client=ChatGithub(...))

def server(input, output, session):
    chat_session = qc.client()

    @reactive.effect
    @reactive.event(input.querychat_user_input)
    async def _():
        query = input.querychat_user_input()
        chunks = retrieve(query, top_k=3)
        if chunks:
            augmented = f"Relevant context:\\n{'\\n\\n'.join(chunks)}\\n\\nQuestion: {query}"
            chat_session.chat(augmented, echo="none")
```

`retrieve()` is fast (TF-IDF, no API) — adds < 1ms per query.